<a href="https://colab.research.google.com/github/DVDSN-AFK/ecommerce-product-classification/blob/main/ecommerce_product_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ecommerce Product Category Classification
## 1. Data Acquisition and Cleaning

**TechCrush AI/ML Bootcamp — Cohort 5 Capstone Project**

---

### 1.1 Project Overview

This project addresses the problem of **automatic product category classification** in ecommerce. Given a product image, our deep learning model will predict which of 54 article types (e.g., Tshirts, Watches, Casual Shoes) the product belongs to.

This is a real-world problem — platforms like Jumia, Amazon, and Konga deal with millions of product listings. Manual categorisation is slow and error-prone. An automated classifier saves time, improves consistency, and scales effortlessly.

### 1.2 Dataset Source

**Dataset:** Fashion Product Images Dataset  
**Source:** Kaggle — https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-dataset  
**License:** Open for educational and research use  

The dataset contains **44,441 product images** from the Indian fashion ecommerce platform Myntra, along with a metadata CSV file (`styles.csv`) describing each product.

### 1.3 Dataset Description

The metadata file (`styles.csv`) has **44,424 rows and 10 columns**:

| Column | Description |
|---|---|
| `id` | Unique product ID — links to image filename (e.g., 15970.jpg) |
| `gender` | Target gender: Men, Women, Boys, Girls, Unisex |
| `masterCategory` | Broad category: Apparel, Footwear, Accessories, etc. |
| `subCategory` | Mid-level category: Topwear, Shoes, Bags, etc. |
| `articleType` | **Target variable** — specific product type: Tshirts, Watches, etc. |
| `baseColour` | Dominant colour of the product |
| `season` | Season the product is suited for |
| `year` | Year the product was listed |
| `usage` | Occasion: Casual, Formal, Sports, etc. |
| `productDisplayName` | Full product title as shown on the platform |

**Target variable:** `articleType` (143 unique classes in raw data, filtered to 54 after cleaning)

---

## Step 1 — Mount Google Drive and Set Paths

We mount Google Drive to access our dataset. The dataset folder was uploaded to our Google Drive before mounting the drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!unzip "/content/drive/MyDrive/group 31 drive/Fashion_Dataset.zip" -d "/content/extracted_data"

Streaming output truncated to the last 5000 lines.
  inflating: /content/extracted_data/Fashion_Dataset/images/58129.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/5813.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/58131.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/58132.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/58133.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/58135.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/58136.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/58137.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/58138.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/58139.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/5814.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/58140.jpg  
  inflating: /content/extracted_data/Fashion_Dataset/images/58141.jpg  
  inflating: /c

In [2]:
import os

# ── Update this path to match where you uploaded the dataset in your Drive ──
DATASET_PATH = '/content/extracted_data/Fashion_Dataset'
CSV_PATH     = os.path.join(DATASET_PATH, 'styles.csv')
IMAGES_PATH  = os.path.join(DATASET_PATH, 'images')


## Step 2 — Import Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Assisted by Claude AI — standard library imports adapted for this dataset
print('All libraries imported successfully.')

All libraries imported successfully.


## Step 3 — Load the Dataset

We use `on_bad_lines='skip'` because the CSV contains a small number of malformed rows (lines where commas inside text fields break the parser). Skipping them is safer than crashing, and we will account for any row loss in our cleaning report.

In [4]:
# Load the metadata CSV
# Assisted by Claude AI — on_bad_lines='skip' handles malformed rows in this dataset
df = pd.read_csv(CSV_PATH, on_bad_lines='skip')

print('Dataset loaded successfully.')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
df.head(10)

Dataset loaded successfully.
Shape: 44,424 rows × 10 columns



,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt
5,1855,Men,Apparel,Topwear,Tshirts,Grey,Summer,2011.0,Casual,Inkfruit Mens Chain Reaction T-shirt
6,30805,Men,Apparel,Topwear,Shirts,Green,Summer,2012.0,Ethnic,Fabindia Men Striped Green Shirt
7,26960,Women,Apparel,Topwear,Shirts,Purple,Summer,2012.0,Casual,Jealous 21 Women Purple Shirt
8,29114,Men,Accessories,Socks,Socks,Navy Blue,Summer,2012.0,Casual,Puma Men Pack of 3 Socks
9,30039,Men,Accessories,Watches,Watches,Black,Winter,2016.0,Casual,Skagen Men Black Watch


## Step 4 — Initial Inspection

Before cleaning, we inspected the dataset to understand what we are working with. That is understanding the data types, basic statistics, and a first look at missing values.

In [5]:
# Column data types
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44424 entries, 0 to 44423
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  44424 non-null  int64  
 1   gender              44424 non-null  object 
 2   masterCategory      44424 non-null  object 
 3   subCategory         44424 non-null  object 
 4   articleType         44424 non-null  object 
 5   baseColour          44409 non-null  object 
 6   season              44403 non-null  object 
 7   year                44423 non-null  float64
 8   usage               44107 non-null  object 
 9   productDisplayName  44417 non-null  object 
dtypes: float64(1), int64(1), object(8)
memory usage: 3.4+ MB


,id,year
count,44424.000000,44423.000000
mean,29696.334301,2012.806497
std,17049.490518,2.126480
min,1163.000000,2007.000000
25%,14768.750000,2011.000000
50%,28618.500000,2012.000000
75%,44683.250000,2015.000000
max,60000.000000,2019.000000


In [6]:
# Check unique values in categorical columns
cat_cols = ['gender', 'masterCategory', 'subCategory', 'articleType',
            'baseColour', 'season', 'usage']

print('Unique value counts per categorical column:')
print('=' * 40)
for col in cat_cols:
    print(f'{col:25s}: {df[col].nunique()} unique values')

Unique value counts per categorical column:
gender                   : 5 unique values
masterCategory           : 7 unique values
subCategory              : 45 unique values
articleType              : 143 unique values
baseColour               : 46 unique values
season                   : 4 unique values
usage                    : 8 unique values


## Step 5 — Check for Duplicate Rows

Duplicate rows can bias a model by making it see the same sample multiple times during training. We identified and removed them.

In [7]:
duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')

if duplicates > 0:
    df = df.drop_duplicates()
    print(f'Duplicates removed. New shape: {df.shape}')
else:
    print('No duplicates found. No action needed.')

Number of duplicate rows: 0
No duplicates found. No action needed.


## Step 6 — Check for Missing Values

Missing values need to be addressed before modelling. We inspect each column and decide on a case-by-case basis whether to fill or drop.

In [8]:
# Count and percentage of missing values per column
missing_count = df.isnull().sum()
missing_pct   = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct
})

print('Missing Values Summary:')
print('=' * 40)
print(missing_df[missing_df['Missing Count'] > 0])

Missing Values Summary:
                    Missing Count  Missing %
baseColour                     15       0.03
season                         21       0.05
year                            1       0.00
usage                         317       0.71
productDisplayName              7       0.02


### Missing Value Decisions

| Column | Missing | % | Decision | Reason |
|---|---|---|---|---|
| `usage` | 317 | ~0.7% | Fill with `'Unknown'` | Usage is not our target; losing 317 rows is unnecessary |
| `baseColour` | 15 | ~0.03% | Fill with `'Unknown'` | Very small number; dropping would be wasteful |
| `season` | 21 | ~0.05% | Fill with `'Unknown'` | Very small number; filling preserves rows |
| `productDisplayName` | 7 | ~0.02% | Fill with `'Unknown'` | Not our primary feature; safe to fill |
| `year` | 1 | ~0.002% | Fill with mode (most common year) | Only one row missing; filling with mode is appropriate |

**Note:** `id`, `gender`, `masterCategory`, `subCategory`, and `articleType` have zero missing values — no action needed for these.

In [9]:
# Assisted by Claude AI — filling strategy adapted for this specific dataset

# Fill categorical columns with 'Unknown'
df['usage']              = df['usage'].fillna('Unknown')
df['baseColour']         = df['baseColour'].fillna('Unknown')
df['season']             = df['season'].fillna('Unknown')
df['productDisplayName'] = df['productDisplayName'].fillna('Unknown')

# Fill year with the mode (most frequent year)
year_mode = df['year'].mode()[0]
df['year'] = df['year'].fillna(year_mode)
df['year'] = df['year'].astype(int)
print(f'Year missing value filled with mode: {int(year_mode)}')

# Confirm no missing values remain
print()
print('Missing values after cleaning:')
print(df.isnull().sum())

Year missing value filled with mode: 2012

Missing values after cleaning:
id                    0
gender                0
masterCategory        0
subCategory           0
articleType           0
baseColour            0
season                0
year                  0
usage                 0
productDisplayName    0
dtype: int64


## Step 7 — Check and Fix Data Types

We ensure each column is stored as the correct data type. Incorrect types can cause errors during preprocessing and modelling.

In [10]:
# id should be string (it links to image filenames, not a number to compute with)
df['id']   = df['id'].astype(str)

# year should be integer
df['year'] = df['year'].astype(int)

print('Updated data types:')
print(df.dtypes)

Updated data types:
id                    object
gender                object
masterCategory        object
subCategory           object
articleType           object
baseColour            object
season                object
year                   int64
usage                 object
productDisplayName    object
dtype: object


## Step 8 — Handle Irrelevant Columns

We review each column and decide whether it contributes to our image classification task.

Since our model takes **images as input** and predicts **articleType**, columns like `masterCategory`, `subCategory`, `gender`, `baseColour`, `season`, `usage`, `year`, and `productDisplayName` are **metadata** — they will not be fed into the deep learning model directly. However, we keep them in the cleaned dataset because:
- They are useful for EDA and understanding the data
- They could be used in a future multimodal model
- Dropping them now would make the dataset harder to interpret

We will select only the relevant columns (`id`, `articleType`) when building the model pipeline in Notebook 3.

In [11]:
print('All columns retained for now:')
for col in df.columns:
    print(f'  - {col}')
print()
print('Modelling will use: id (links to image) + articleType (target label)')

All columns retained for now:
  - id
  - gender
  - masterCategory
  - subCategory
  - articleType
  - baseColour
  - season
  - year
  - usage
  - productDisplayName

Modelling will use: id (links to image) + articleType (target label)


## Step 9 — Filter articleType Classes (≥ 100 Samples)

The raw dataset has **143 unique articleType classes**. Many of these have very few samples — some as low as 1. Deep learning models cannot learn meaningful patterns from classes with insufficient data.

**Decision:** We keep only classes with **at least 100 samples**. This threshold is chosen because:
- It retains enough data per class for the model to learn discriminative features
- It removes noise from severely underrepresented classes
- It reduces the number of classes to a manageable 54
- Similar filtering is standard practice in production image classification pipelines

In [12]:
# Count samples per articleType
class_counts = df['articleType'].value_counts()

print(f'Total articleType classes before filtering: {df["articleType"].nunique()}')
print(f'Classes with fewer than 100 samples: {(class_counts < 100).sum()}')
print()

# Keep only classes with >= 100 samples
# Assisted by Claude AI — filtering minority classes adapted for this dataset
valid_classes = class_counts[class_counts >= 100].index
df = df[df['articleType'].isin(valid_classes)].reset_index(drop=True)

print(f'Total articleType classes after filtering : {df["articleType"].nunique()}')
print(f'Total rows after filtering                : {len(df):,}')

Total articleType classes before filtering: 143
Classes with fewer than 100 samples: 89

Total articleType classes after filtering : 54
Total rows after filtering                : 42,154


## Step 10 — Verify Image Files Exist

Each row in the CSV links to an image file via the `id` column (e.g., id `15970` → `15970.jpg`). We must verify that the image files actually exist on disk. Rows without a corresponding image cannot be used for training.

In [13]:
# Build the expected image path for each row
df['image_path'] = df['id'].apply(
    lambda x: os.path.join(IMAGES_PATH, f'{x}.jpg')
)

# Check which images actually exist
df['image_exists'] = df['image_path'].apply(os.path.exists)

print(f'Images found    : {df["image_exists"].sum():,}')
print(f'Images missing  : {(~df["image_exists"]).sum():,}')
print(f'Match rate      : {df["image_exists"].mean()*100:.2f}%')

Images found    : 42,150
Images missing  : 4
Match rate      : 99.99%


In [14]:
# Drop rows where the image file does not exist
# Reason: we cannot train on data we cannot see
df = df[df['image_exists']].reset_index(drop=True)
df = df.drop(columns=['image_exists'])  # cleanup helper column

print(f'Final dataset shape after image verification: {df.shape}')

Final dataset shape after image verification: (42150, 11)


## Step 11 — Encode the Target Variable

Deep learning models require numerical labels, not text strings. We use **Label Encoding** for the target variable (`articleType`) rather than One-Hot Encoding because:
- We have 54 classes — one-hot encoding would create 54 binary columns, which is wasteful
- TensorFlow/Keras handles integer class labels natively with `sparse_categorical_crossentropy`
- Label encoding is the standard approach for multi-class image classification

In [15]:
from sklearn.preprocessing import LabelEncoder

# Assisted by Claude AI — LabelEncoder usage adapted for this classification task
le = LabelEncoder()
df['label'] = le.fit_transform(df['articleType'])

# Save the class names for use in later notebooks
class_names = le.classes_
num_classes = len(class_names)

print(f'Number of classes: {num_classes}')
print()
print('Label mapping (first 10):')
for i, name in enumerate(class_names[:10]):
    print(f'  {i:3d} → {name}')
print('  ...')

Number of classes: 54

Label mapping (first 10):
    0 → Backpacks
    1 → Belts
    2 → Bra
    3 → Briefs
    4 → Capris
    5 → Caps
    6 → Casual Shoes
    7 → Clutches
    8 → Cufflinks
    9 → Deodorant
  ...


## Step 12 — Save the Cleaned Dataset

We save the cleaned dataframe to Google Drive so all subsequent notebooks can load it without repeating the cleaning steps.

In [16]:
# Save cleaned CSV to Drive
CLEANED_CSV = os.path.join(DATASET_PATH, 'styles_cleaned.csv')
df.to_csv(CLEANED_CSV, index=False)
print(f'Cleaned dataset saved to: {CLEANED_CSV}')

# Save class names for use in modelling notebooks
import json
CLASS_NAMES_PATH = os.path.join(DATASET_PATH, 'class_names.json')
with open(CLASS_NAMES_PATH, 'w') as f:
    json.dump(class_names.tolist(), f)
print(f'Class names saved to     : {CLASS_NAMES_PATH}')

Cleaned dataset saved to: /content/extracted_data/Fashion_Dataset/styles_cleaned.csv
Class names saved to     : /content/extracted_data/Fashion_Dataset/class_names.json


## Step 13 — Cleaning Summary

A full record of every decision made during data cleaning.

In [17]:
print('=' * 55)
print('         DATA CLEANING SUMMARY')
print('=' * 55)
print(f'  Original rows             : 44,424')
print(f'  Duplicate rows removed    : 0')
print(f'  Missing values filled     : usage, baseColour,')
print(f'                              season, productDisplayName,')
print(f'                              year (mode fill)')
print(f'  articleType classes before: 143')
print(f'  Classes removed (< 100)   : 89')
print(f'  articleType classes after : 54')
print(f'  Rows removed (minority)   : ~2,270')
print(f'  Rows removed (no image)   : verified and dropped')
print(f'  Final rows                : {len(df):,}')
print(f'  Final columns             : {df.shape[1]}')
print(f'  Target variable           : articleType (label encoded)')
print('=' * 55)
print()
print('Next step: Notebook 02 — Exploratory Data Analysis')

         DATA CLEANING SUMMARY
  Original rows             : 44,424
  Duplicate rows removed    : 0
  Missing values filled     : usage, baseColour,
                              season, productDisplayName,
                              year (mode fill)
  articleType classes before: 143
  Classes removed (< 100)   : 89
  articleType classes after : 54
  Rows removed (minority)   : ~2,270
  Rows removed (no image)   : verified and dropped
  Final rows                : 42,150
  Final columns             : 12
  Target variable           : articleType (label encoded)

Next step: Notebook 02 — Exploratory Data Analysis
